In [2]:
# =========================================================
# [VALIDATION] 5-Fold(층화) CV + PCA k 선택 + Ridge alpha 튜닝     # ★ 핵심
# - fold 분할은 tasting_category_fine(스타일) 기준 층화           # ★ 중요
# - scaler/PCA는 각 fold의 train에서만 fit → val에는 transform만  # ★ 누수 방지
# =========================================================

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# =========================
# 0) 설정
# =========================
K_FOLDS = 5                           # fold 수(=k-fold의 k)
K_FIXED = 5                           # PC 개수 k=5 "고정" 실험

max_k = min(X_train.shape[0]-1, X_train.shape[1])  # <= 174 (train=175면)
K_GRID = [2, 5, 10, 20, 30, 40, 60, 80, 120, max_k]
K_GRID = sorted(set([k for k in K_GRID if 1 <= k <= max_k]))

ALPHA_GRID = np.logspace(-4, 4, 17)

# OUT_DIR, RUN_TAG가 기존 노트북에 있으면 그대로 사용
# 없으면 아래 두 줄만 켜서 사용
OUT_DIR = "/home/a202192020/맥주데이터실험/pca/0220/beer_pca_validation_output"
RUN_TAG = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

os.makedirs(OUT_DIR, exist_ok=True)
CV_GRID_CSV    = os.path.join(OUT_DIR, f"cv_grid_{RUN_TAG}.csv")
CV_SUMMARY_CSV = os.path.join(OUT_DIR, f"cv_summary_{RUN_TAG}.csv")
BEST_MODEL_JOB = os.path.join(OUT_DIR, f"best_model_{RUN_TAG}.joblib")

# =========================
# 1) metric / scorer
# =========================
def rmse_multi(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred, multioutput="uniform_average")))

def r2_multi(y_true, y_pred) -> float:
    return float(r2_score(y_true, y_pred, multioutput="uniform_average"))

def neg_rmse_scorer(estimator, X, y):
    pred = estimator.predict(X)
    return -rmse_multi(y, pred)

# =========================
# 2) 5-fold "층화" 분할 만들기
#    - y(관능 50개)는 연속값이라 StratifiedKFold에 직접 못 씀
#    - 대신 style(tasting_category_fine)로 fold를 층화함
# =========================
style_train = train_df[STRATIFY_COL]  # 예: "tasting_category_fine"
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_splits = list(skf.split(X_train, style_train))

# =========================
# 3) (A) Baseline CV: PCA 없이 Ridge
# =========================
baseline_pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
])

scores_base = cross_val_score(
    baseline_pipe, X_train, y_train,
    scoring=neg_rmse_scorer, cv=cv_splits, n_jobs=-1
)
print(f"[CV Baseline] RMSE(mean)={-scores_base.mean():.6f}  std={scores_base.std():.6f}")

# =========================
# 4) (B) PCR fixed: PCA k=5 고정 CV
# =========================
pcr_fixed_pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("pca", PCA(n_components=K_FIXED, svd_solver="full")),
    ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
])

scores_fixed = cross_val_score(
    pcr_fixed_pipe, X_train, y_train,
    scoring=neg_rmse_scorer, cv=cv_splits, n_jobs=-1
)
print(f"[CV PCR fixed k={K_FIXED}] RMSE(mean)={-scores_fixed.mean():.6f}  std={scores_fixed.std():.6f}")

# =========================
# 5) (C) GridSearch: k(PC개수) + alpha(Ridge) 선택
# =========================
pcr_pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ("pca", PCA(svd_solver="full")),
    ("model", Ridge(random_state=RANDOM_STATE)),
])

param_grid = {
    "pca__n_components": K_GRID,
    "model__alpha": ALPHA_GRID,
}

gs = GridSearchCV(
    estimator=pcr_pipe,
    param_grid=param_grid,
    scoring=neg_rmse_scorer,   # negRMSE 최대화 = RMSE 최소화
    cv=cv_splits,
    n_jobs=-1,
    verbose=2,
    refit=True,
)

gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print("Best CV RMSE:", -gs.best_score_)

cv_df = pd.DataFrame(gs.cv_results_)
cv_df.to_csv(CV_GRID_CSV, index=False)
print("✅ saved:", CV_GRID_CSV)

best_pipe = gs.best_estimator_
joblib.dump(best_pipe, BEST_MODEL_JOB)
print("✅ saved:", BEST_MODEL_JOB)

# =========================
# 6) Best 모델로 train/test 평가
# =========================
pred_tr = best_pipe.predict(X_train)
pred_te = best_pipe.predict(X_test)

train_rmse = rmse_multi(y_train, pred_tr)
test_rmse  = rmse_multi(y_test, pred_te)
train_r2   = r2_multi(y_train, pred_tr)
test_r2    = r2_multi(y_test, pred_te)

print("[BEST] Train RMSE:", train_rmse, "R2:", train_r2)
print("[BEST] Test  RMSE:", test_rmse,  "R2:", test_r2)

summary = pd.DataFrame([{
    "run_tag": RUN_TAG,
    "random_state": RANDOM_STATE,
    "n_folds": K_FOLDS,
    "best_k": best_pipe.named_steps["pca"].n_components_,
    "best_alpha": best_pipe.named_steps["model"].alpha,
    "cv_best_rmse": -gs.best_score_,
    "train_rmse": train_rmse,
    "train_r2": train_r2,
    "test_rmse": test_rmse,
    "test_r2": test_r2,
}])
summary.to_csv(CV_SUMMARY_CSV, index=False)
print("✅ saved:", CV_SUMMARY_CSV)

NameError: name 'X_train' is not defined